title: "Denmark: Housing Market Supply"author: "ckrusemd"date: "`r Sys.Date()`"output: html_document

In [ ]:
knitr::opts_chunk$set(echo = FALSE,message = FALSE,warning = FALSE)options(scipen=999)

In [ ]:
if (!require(pacman)) { install.packages("pacman") }pacman::p_load(tidyr,               dplyr,               ggplot2,               boot,               openxlsx,               lubridate,               forcats,               broom,               purrr,               caret,               glue,               devtools,               TTR,               dkstat,               httr,               rvest,               zoo,               rvest)source(file = "money_theme.R")

In [ ]:
gam_adjust = function(data) {    grid <- expand.grid(span = seq(0.1, 0.5, len = 10), degree = c(1))  # fit_gam = suppressWarnings(expr =  { train(y = data$VALUE,   #       x = data %>% dplyr::select(DATE),  #       tuneGrid=grid,  #       method = "gamLoess") })    fit_smooth = suppressWarnings(expr =  { caret::train(y = data$VALUE,                             x = data %>% dplyr::select(DATE),                             method="gamLoess",                            tuneGrid=grid,                            trControl=trainControl(method = "repeatedcv",number = 5,repeats = 1)) })  data %>%     dplyr::mutate(VALUE_SMOOTH=predict(fit_smooth,.)) %>%     dplyr::mutate(VALUE_SMOOTH_SCALED=scale(VALUE_SMOOTH,center=TRUE,scale=TRUE)) %>%    dplyr::mutate(VALUE_SCALED=scale(VALUE,center=TRUE,scale=TRUE)) %>%    dplyr::mutate(VALUE_EMA=TTR::EMA(VALUE,20)) %>%     arrange(desc(DATE)) %>%     dplyr::mutate(VALUE_CHG=VALUE-lead(VALUE)) %>%     filter(!is.na(VALUE_CHG)) %>%     arrange((DATE)) %>%     dplyr::mutate(VALUE_SUM_CHG=cumsum(VALUE_CHG)) %>%     dplyr::mutate(VALUE_SUM_CHG_SCALED=scale(VALUE_SUM_CHG,center=TRUE,scale=TRUE))  }

# Construction (DK)

In [ ]:
BYGV80_meta <- dst_meta(table = "BYGV80", lang = "da")BYGV80 <- dst_get_data(table = "BYGV80",                        BYGFASE="*",                       ANVENDELSE="*",                       Tid="*",                       lang = "da") %>%   group_by(BYGFASE,ANVENDELSE) %>%   arrange(BYGFASE,ANVENDELSE,TID) %>%   filter(ANVENDELSE %in% c("Etageboliger","Parcelhuse")) %>%   dplyr::mutate(ANVENDELSE=gsub("Etageboliger","FLAT",ANVENDELSE)) %>%   dplyr::mutate(ANVENDELSE=gsub("Parcelhuse","HOUSE",ANVENDELSE)) %>%   dplyr::rename(PROPERTY_TYPE=ANVENDELSE) %>%   dplyr::rename(STAT=BYGFASE) %>%   dplyr::rename(DATE=TID) %>%   dplyr::rename(VALUE=value) %>%   group_by(PROPERTY_TYPE,STAT) %>%   dplyr::mutate(PROPERTY_TYPE=recode(PROPERTY_TYPE, 'HOUSE'='Parcel-/rækkehuse', 'FLAT'='Ejerlejlighed')) %>%   do(gam_adjust(.))BYG1_meta <- dst_meta(table = "BYG1", lang = "da")BYG1 <- dst_get_data(table = "BYG1",                        BRANCHE07="*",                       SÆSON="*",                       ART="*",                       Tid="*",                       lang = "da") %>%   dplyr::rename(DATE=TID) %>%   dplyr::rename(VALUE=value) %>%   filter(BRANCHE07=="F Bygge og anlæg") %>%   filter(SÆSON=="Sæsonkorrigeret") %>%   filter(ART %in% c("I alt","Nybyggeri og tilbygning i alt","Reparation og vedligeholdelse i alt")) %>%   group_by(BRANCHE07,SÆSON,ART) %>%   arrange(BRANCHE07,SÆSON,ART,DATE) %>%   do(gam_adjust(.))PRIS90_meta <- dst_meta(table = "PRIS90", lang = "da")PRIS90 <- dst_get_data(table = "PRIS90",                        ENHED="*",                       BOLTYP="*",                       Tid="*",                       lang = "da") %>%   dplyr::rename(DATE=TID) %>%   dplyr::rename(VALUE=value) %>%   filter(ENHED=="Indeks") %>%   do(gam_adjust(.))BYG42_meta <- dst_meta(table = "BYG42", lang = "da")BYG42 <- dst_get_data(table = "BYG42",                        HINDEKS="*",                       DINDEKS="*",                       ART="*",                       TAL="*",                       Tid="*",                       lang = "da") %>%   dplyr::rename(DATE=TID) %>%   dplyr::rename(VALUE=value) %>%   filter(DINDEKS=="Byggeomkostningsindeks i alt") %>%   filter(TAL=="Indeks") %>%   filter(ART %in% c("Materialer","Arbejdsomkostninger")) %>%   group_by( HINDEKS , DINDEKS , ART , TAL ) %>%   do(gam_adjust(.))

In [ ]:
BYGV80 %>%   filter(PROPERTY_TYPE=="Parcel-/rækkehuse") %>%   ggplot(.,aes(x=DATE,y=VALUE_SUM_CHG_SCALED,color=STAT)) +  geom_line() +   # facet_wrap(~STAT,scales="free") +  # geom_line(aes(y=VALUE_SCALED),alpha=0.3) +  theme_money_printer_go_brrr(base_size=12) +  geom_hline(yintercept = 0,linetype=2)

In [ ]:
df_ccf_BYGV80 = BYGV80 %>%   filter(PROPERTY_TYPE=="Parcel-/rækkehuse") %>%   dplyr::select(STAT,PROPERTY_TYPE,DATE,VALUE_SMOOTH) %>%   spread(STAT,VALUE_SMOOTH)ccf_results = ccf(df_ccf_BYGV80$`Tilladt byggeri`,df_ccf_BYGV80$`Fuldført byggeri`,lag.max = 60,plot = FALSE)  plot(ccf_results)  best_corr_index = which(abs(ccf_results$acf)==max(abs(ccf_results$acf)))  ccf_results$lag[best_corr_index]  ccf_results$acf[best_corr_index]

In [ ]:
df_ccf_BYG1 = BYGV80 %>%   filter(STAT=="Fuldført byggeri") %>%   filter(PROPERTY_TYPE=="Parcel-/rækkehuse") %>%   ungroup() %>%   dplyr::select(STAT,DATE,VALUE_SMOOTH) %>%   bind_rows(        BYG1 %>%       filter(BRANCHE07=="F Bygge og anlæg") %>%       filter(ART=="I alt") %>%       ungroup() %>%       dplyr::select(ART,DATE,VALUE_SMOOTH) %>%       dplyr::rename(STAT=ART)      ) %>%   spread(STAT,VALUE_SMOOTH) %>%   na.omit()ccf_results = ccf(df_ccf_BYG1$`Fuldført byggeri`,df_ccf_BYG1$`I alt`,lag.max = 60,plot = FALSE)  plot(ccf_results)  best_corr_index = which(abs(ccf_results$acf)==max(abs(ccf_results$acf)))  ccf_results$lag[best_corr_index]  ccf_results$acf[best_corr_index]

In [ ]:
BYGV80 %>%   filter(PROPERTY_TYPE=="Parcel-/rækkehuse") %>%   ggplot(.,aes(x=DATE,y=VALUE_SMOOTH_SCALED,color=STAT)) +  geom_line() +  geom_line(aes(y=VALUE_SCALED),alpha=0.3) +  theme_money_printer_go_brrr(base_size=12)

In [ ]:
BYG1 %>%  ggplot(.,aes(x=DATE,y=VALUE_SMOOTH_SCALED,color=ART)) +  geom_point(aes(y=VALUE_SCALED)) +  geom_smooth(span=0.17) +  theme(legend.position = "bottom") +  geom_hline(yintercept = 0,linetype=2)  +  theme_money_printer_go_brrr(base_size=12) +  labs(title="Denmark: Construction Employment",       x=NULL,       y="(SD)",       caption = timestamp_caption())

In [ ]:
BYGV80 %>%   filter(PROPERTY_TYPE=="Parcel-/rækkehuse") %>%   dplyr::select(PROPERTY_TYPE,STAT,DATE,VALUE_SMOOTH) %>%   spread(STAT,VALUE_SMOOTH)BYGV80 %>%   filter(PROPERTY_TYPE %in% c("Parcel-/rækkehuse")) %>%   group_by(STAT) %>%   arrange(STAT,DATE)  %>%   filter(STAT %in% c("Tilladt byggeri","Fuldført byggeri","Påbegyndt byggeri","Byggeri under opførelse")) %>%   dplyr::mutate(VALUE_SMOOTH=cumsum(VALUE_SMOOTH)) %>%   dplyr::select(PROPERTY_TYPE,STAT,DATE,VALUE_SMOOTH) %>%   spread(STAT,VALUE_SMOOTH) %>%   dplyr::mutate(backlogTP=`Tilladt byggeri`-`Påbegyndt byggeri`) %>%   dplyr::mutate(backlogTF=`Tilladt byggeri`-`Fuldført byggeri`) %>%   dplyr::mutate(backlogPF=`Påbegyndt byggeri`-`Fuldført byggeri`) %>%   # dplyr::mutate(backlogTB=`Byggeri under opførelse`-`Fuldført byggeri`) %>%   gather(backlog_type,backlog_value,backlogTP,backlogTF,backlogPF) %>%   # dplyr::mutate(backlog=scale(backlog,center=TRUE,scale=TRUE)) %>%   ggplot(.,aes(x=DATE,y=backlog_value,color=backlog_type)) +  geom_point(size=0.7) +  # geom_smooth(span=0.25) +  theme(legend.position = "bottom") +  geom_hline(yintercept = 0,linetype=2) +  theme_money_printer_go_brrr(base_size=12) +  geom_vline(xintercept = c(ymd("2005-04-01","2021-01-01"))) +  geom_vline(xintercept = c(ymd("2007-06-01","2022-03-31")),linetype=3) +  labs(title="Denmark: Backlog of Construction (Permits % Completed)",       x=NULL,       y="(SD)",       caption = timestamp_caption()) +  scale_x_date(date_breaks = "1 year",date_labels = "%Y")

In [ ]:
df_fp = BYGV80 %>%   filter(PROPERTY_TYPE %in% c("Parcel-/rækkehuse")) %>%   group_by(STAT) %>%   arrange(STAT,DATE)  %>%   filter(STAT %in% c("Tilladt byggeri","Fuldført byggeri","Påbegyndt byggeri","Byggeri under opførelse")) %>%   dplyr::mutate(VALUE_SMOOTH=cumsum(VALUE)) %>%   dplyr::select(PROPERTY_TYPE,STAT,DATE,VALUE_SMOOTH) %>%   spread(STAT,VALUE_SMOOTH) %>%   dplyr::mutate(backlogTF=`Tilladt byggeri`-`Fuldført byggeri`) %>%   # filter(DATE>=ymd("2021-04-01"))  %>%   filter(DATE>=ymd("2022-01-01")) # lm_poly2 = lm(df_fp$backlogTF ~ poly(df_fp$DATE,2))lm_poly2 = lm( backlogTF ~ poly(DATE,2) ,data=df_fp)summary(lm_poly2)df_fp_extrapolate = data.frame(DATE=seq(ymd("2021-02-01"),ymd("2024-01-01"),by="1 month")) %>%   dplyr::mutate(backlogTF=predict(lm_poly2,.)) df_fp %>%   ggplot(.,aes(x=DATE,y=backlogTF)) +  geom_point() +  geom_hline(yintercept = 0,linetype=2)  +  geom_vline(xintercept = ymd("2023-05-01")) +  scale_x_date(date_breaks = "1 year",date_labels = "%Y",limits = c(ymd("2020-01-01"),ymd("2025-01-01"))) +  geom_line(data=df_fp_extrapolate,aes(x=DATE,y=backlogTF))

# Denmark Housing Supply

In [ ]:
df_udbud =  openxlsx::read.xlsx(xlsxFile = "Data/UDB010.xlsx",startRow = 3) %>%   dplyr::rename(SUPPLY_TYPE=X2,                PROPERTY_TYPE=X1,                ZIP=X3) %>%   tidyr::fill(SUPPLY_TYPE,.direction="down") %>%   tidyr::fill(PROPERTY_TYPE,.direction="down") %>%   tidyr::fill(ZIP,.direction="down") %>%   gather(DATE,VALUE,c(4:ncol(.))) %>%   filter(!VALUE=="..") %>%   dplyr::mutate(DATE=lubridate::ceiling_date(ymd(glue("{substr(DATE,1,4)}-{substr(DATE,6,7)}-01")),unit="month")-days(1)) %>%   group_by(PROPERTY_TYPE,SUPPLY_TYPE,ZIP) %>%  do(loess_adjust(.))

In [ ]:
df_udbud.change =  openxlsx::read.xlsx(xlsxFile = "Data/UDB010.xlsx",startRow = 3) %>%   dplyr::rename(SUPPLY_TYPE=X2,                PROPERTY_TYPE=X1,                ZIP=X3) %>%   tidyr::fill(SUPPLY_TYPE,.direction="down") %>%   tidyr::fill(PROPERTY_TYPE,.direction="down") %>%   tidyr::fill(ZIP,.direction="down") %>%   gather(DATE,VALUE,c(4:ncol(.))) %>%   filter(!VALUE=="..") %>%   dplyr::mutate(DATE=lubridate::ceiling_date(ymd(glue("{substr(DATE,1,4)}-{substr(DATE,6,7)}-01")),unit="month")-days(1)) %>%   group_by(PROPERTY_TYPE,SUPPLY_TYPE,ZIP) %>%    do(loess_adjust(.)) %>%   arrange(PROPERTY_TYPE,ZIP,DATE) %>%   dplyr::mutate(next_period_chg=lead(VALUE)-VALUE)

In [ ]:
df_udbud.bubblelow = df_udbud %>%   filter(DATE>ymd("2005-01-01")) %>%   filter(DATE<ymd("2008-01-01")) %>%   group_by(PROPERTY_TYPE,ZIP) %>%   filter(VALUE==min(VALUE)) %>%   dplyr::select(PROPERTY_TYPE,ZIP,DATE)df_udbud.bubblelow.DK =   df_udbud.bubblelow %>%   group_by(PROPERTY_TYPE) %>%   dplyr::summarise(DATE=mean(DATE))

In [ ]:
df_udbud.covid19low = df_udbud %>%   filter(DATE>ymd("2020-01-01")) %>%   group_by(PROPERTY_TYPE,ZIP) %>%   filter(VALUE==min(VALUE)) %>%   dplyr::select(PROPERTY_TYPE,ZIP,DATE)df_udbud.covid19low.DK =   df_udbud.covid19low %>%   group_by(PROPERTY_TYPE) %>%   dplyr::summarise(DATE=mean(DATE))

In [ ]:
df.house_prices =  openxlsx::read.xlsx(xlsxFile = "Data/BM010.xlsx",startRow = 3) %>%   rename_with(~c("PROPERTY_TYPE","SALES_TYPE","ZIP"),1:3) %>%   tidyr::fill(PROPERTY_TYPE,.direction="down") %>%   tidyr::fill(SALES_TYPE,.direction="down") %>%   gather(DATE,VALUE,c(4:ncol(.))) %>%   filter(!VALUE=="..") %>%   dplyr::mutate(VALUE=as.integer(VALUE)) %>%   dplyr::mutate(DATE=lubridate::ceiling_date(as.Date(as.yearqtr(DATE, format = "%YK%q")),"quarters")-days(1)) %>%   group_by(PROPERTY_TYPE,SALES_TYPE,ZIP) %>%   # do(loess_adjust(.)) %>%   dplyr::mutate(PROPERTY_TYPE=recode(PROPERTY_TYPE,                                     'HOUSE'='Parcel-/rækkehuse',                                     'Parcel-/rækkehus'='Parcel-/rækkehuse',                                     'FLAT'='Ejerlejlighed'))

In [ ]:
df_liggetider =  openxlsx::read.xlsx(xlsxFile = "Data/UDB030.xlsx",startRow = 3) %>%   dplyr::rename(STAT=X2,                PROPERTY_TYPE=X1,                ZIP=X3) %>%   tidyr::fill(STAT,.direction="down") %>%   tidyr::fill(PROPERTY_TYPE,.direction="down") %>%   filter(STAT=="Liggetider (dage)") %>%   gather(DATE,VALUE,c(4:ncol(.))) %>%   filter(!VALUE=="..") %>%   group_by(PROPERTY_TYPE,STAT,ZIP) %>%   dplyr::mutate(INDEX=scale(VALUE,center = TRUE,scale = TRUE)) %>%   dplyr::mutate(DATE=lubridate::ceiling_date(ymd(glue("{substr(DATE,1,4)}-{substr(DATE,6,7)}-01")),unit="month")-days(1))

In [ ]:
source_url = "https://finansdanmark.dk/tal-og-data/boligstatistik/obligationsrenter/"current_url = xml2::read_html(source_url) %>%  html_node("body > main > div > div.page-header > div.page-header__content > div > div.row > div.col-12.col-md-8 > div > p:nth-child(11) > a") %>%   rvest::html_attr("href")xlsx_url = paste0("https://finansdanmark.dk/",current_url)df_interest = openxlsx::read.xlsx(xlsxFile = xlsx_url,startRow = 1)df_interest$År[1] = 1997df_interest = df_interest %>% fill(År,.direction = "down")df_interest$Date = as.Date(paste(df_interest$År, df_interest$Uge, 1, sep="-"), "%Y-%U-%u")df_interest = df_interest %>%   dplyr::select(Date,Kort.rente,Lang.rente) %>%   dplyr::mutate( CurveLongShort = Lang.rente - Kort.rente ) %>%   gather(STAT,VALUE,Kort.rente:CurveLongShort) %>%   # dplyr::mutate(VALUE=TTR::RSI(VALUE,n=14)) %>%   dplyr::mutate(INDEX=scale(VALUE,center = TRUE,scale = TRUE)) %>%   dplyr::rename(DATE=Date)

In [ ]:
df.adjustments =   openxlsx::read.xlsx(xlsxFile = "Data/BM010.xlsx",startRow = 3) %>%   rename_with(~c("PROPERTY_TYPE","SALES_TYPE","ZIP"),1:3) %>%   tidyr::fill(PROPERTY_TYPE,.direction="down") %>%   tidyr::fill(SALES_TYPE,.direction="down") %>%   tidyr::fill(ZIP,.direction="down") %>%   gather(DATE,VALUE,c(4:ncol(.))) %>%   filter(!VALUE=="..") %>%   filter(SALES_TYPE=="Lyngby-Taarbæk") %>%   # filter(PROPERTY_TYPE=="Parcel-/rækkehus") %>%   dplyr::mutate(VALUE=as.integer(VALUE)) %>%   unite(PROPERTY_TYPE,PROPERTY_TYPE,ZIP) %>%   spread(PROPERTY_TYPE,VALUE) %>%   dplyr::mutate(ADJUST_HOUSE=(`Parcel-/rækkehus_Realiseret handelspris`/`Parcel-/rækkehus_Første udbudspris`)-1) %>%   # dplyr::mutate(ADJUST_FLAT=(`Realiseret handelspris_FLAT`/`Første udbudspris_FLAT`)-1) %>%   dplyr::select(SALES_TYPE,DATE,ADJUST_HOUSE) %>%   gather(ADJUST_TYPE,VALUE,ADJUST_HOUSE) %>%   na.omit() %>%   separate(ADJUST_TYPE,into=c("STAT","PROPERTY_TYPE")) %>%   dplyr::mutate(DATE=lubridate::ceiling_date(as.Date(as.yearqtr(DATE, format = "%YK%q")),"quarters")-days(1)) %>%   dplyr::mutate(PROPERTY_TYPE=recode(PROPERTY_TYPE, 'HOUSE'='Parcel-/rækkehuse', 'FLAT'='Ejerlejlighed')) %>%   group_by(PROPERTY_TYPE,STAT,SALES_TYPE) %>%   dplyr::mutate(INDEX=scale(VALUE,center = TRUE,scale = TRUE)) 

### All time development

In [ ]:
df.adjustments %>%   bind_rows( df.house_prices ) %>%   bind_rows( df_liggetider ) %>%   bind_rows( df_interest ) %>%   bind_rows( df_BYGV80 ) %>%   filter(!PROPERTY_TYPE=="Ejerlejlighed") %>%  ggplot(.,aes(x=DATE,y=INDEX,color=STAT)) +  geom_smooth(fill=NA,span=0.03) +  facet_wrap(~PROPERTY_TYPE,ncol=1) +  geom_smooth(data=df_udbud,aes(x=DATE,y=VALUE),color="blue",linetype=2)  +  scale_x_date(date_breaks = "1 year",date_labels = "%Y") +   theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1)) +  theme(legend.position="bottom") +  geom_vline(data=df_udbud.covid19low.DK,aes(xintercept=DATE)) +  geom_vline(data=df_udbud.bubblelow.DK,aes(xintercept=DATE)) +  geom_hline(yintercept = 0) +  theme_money_printer_go_brrr(base_size=12) 

In [ ]:
df_BYGV80 %>%   filter(!PROPERTY_TYPE=="Ejerlejlighed") %>%  ggplot(.,aes(x=DATE,y=VALUE_SCALED,color=STAT)) +  geom_line() +  geom_smooth(fill=NA,span=0.1) +  facet_wrap(~PROPERTY_TYPE,ncol=1) +  geom_smooth(data=df_udbud,aes(x=DATE,y=VALUE),color="blue",linetype=2,span=0.1)  +  scale_x_date(date_breaks = "1 year",date_labels = "%Y",limits=c(dmy("2004-01-01"),dmy("2008-01-01"))) +   theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1)) +  theme(legend.position="bottom") +  geom_vline(data=df_udbud.covid19low.DK,aes(xintercept=DATE)) +  geom_vline(data=df_udbud.bubblelow.DK,aes(xintercept=DATE)) +  geom_hline(yintercept = 0)  +  theme_money_printer_go_brrr(base_size=12) 

In [ ]:
df_udbud %>%   ggplot(.,aes(x=DATE,y=VALUE,color=ZIP)) +  geom_line() +  facet_wrap(~PROPERTY_TYPE+SUPPLY_TYPE) +  theme_money_printer_go_brrr(base_size=12) 

### Recent developments

In [ ]:
df_BYGV80 %>%   filter(PROPERTY_TYPE=="Parcel-/rækkehuse") %>%   # filter(DATE>=Sys.Date()-years(10)) %>%   ggplot(.,aes(x=DATE,y=VALUE_SMOOTH,color=STAT)) +  geom_line() +  geom_point(aes(y=VALUE,color=STAT)) +  # geom_point() +  facet_wrap(~STAT,scales="free") +  theme(legend.position="bottom",        axis.text.x=element_text(angle=45,hjust=1)) +  scale_y_continuous(labels = scales::number_format(big.mark=" ")) +  # scale_x_date(date_breaks = "6 months",date_labels = "%Y %b")  +  scale_x_date(date_breaks = "2 years",date_labels = "%Y %b")  +  # geom_vline(xintercept = seq.Date(from = dmy("01-01-2000"),to = dmy("01-01-2030"),by = "1 year"),linetype=2,alpha=0.7)  +  # geom_vline(xintercept = seq.Date(from = dmy("01-08-2000"),to = dmy("01-08-2030"),by = "1 year"),linetype=2,alpha=0.8) +  theme_money_printer_go_brrr(base_size=16) 

In [ ]:
df_BYGV80 %>%   filter(PROPERTY_TYPE=="Parcel-/rækkehuse") %>%   filter(DATE>=Sys.Date()-years(5)) %>%   ggplot(.,aes(x=DATE,y=VALUE,color=STAT)) +  geom_line() +  geom_point() +  facet_wrap(~STAT,scales="free") +  theme(legend.position="bottom") +  scale_y_continuous(labels = scales::number_format(big.mark=" ")) +  scale_x_date(date_breaks = "1 year",date_labels = "%Y") +  theme_money_printer_go_brrr(base_size=16) 

In [ ]:
df_BYGV80 %>%   filter(PROPERTY_TYPE=="Parcel-/rækkehuse") %>%   group_by(STAT) %>%   dplyr::mutate(VALUE=lead(VALUE)-VALUE) %>%   ggplot(.,aes(x=DATE,y=VALUE,color=STAT)) +  geom_line() +  geom_point() +  facet_wrap(~STAT,scales="free") +  theme(legend.position="bottom") +  scale_y_continuous(labels = scales::number_format(big.mark=" ")) +  geom_hline(yintercept = 0) +  theme_money_printer_go_brrr(base_size=16) 

### Permits and house prices

In [ ]:
df.house_prices %>%   filter(ZIP=="Hele landet") %>%   dplyr::rename(STAT=SALES_TYPE) %>%   filter(STAT=="Realiseret handelspris") %>%   dplyr::mutate(MoMchange=VALUE/lag(VALUE)) %>%   dplyr::mutate(INDEX=scale(MoMchange)) %>%    bind_rows( df_BYGV80 ) %>%   # bind_rows( df.adjustments )  filter(PROPERTY_TYPE=="Parcel-/rækkehuse") %>%   filter(STAT %in% c("Tilladt byggeri","Realiseret handelspris")) %>%   ggplot(.,aes(x=DATE,y=INDEX,color=STAT)) +  geom_line() +  scale_x_date(date_breaks = "1 year",date_labels = "%Y") +  theme_money_printer_go_brrr(base_size=16) +  geom_hline(yintercept = 0)# df_cor_price_tilladelse = df.house_prices %>% #   filter(ZIP=="Hele landet") %>% #   dplyr::rename(STAT=SALES_TYPE) %>% #   dplyr::mutate(VALUE=VALUE/lag(VALUE)) %>% #   dplyr::mutate(INDEX=scale(VALUE)) %>% #   bind_rows( df_BYGV80 ) %>% #   filter(STAT %in% c("Tilladt byggeri","Realiseret handelspris")) %>% #   filter(PROPERTY_TYPE=="Parcel-/rækkehuse") %>% #   spread(STAT,INDEX) %>% #   na.omit()# plot(df_cor_price_tilladelse$VALUE,df_cor_price_tilladelse$`Realiseret handelspris`)# cor( df_cor_price_tilladelse$MoMchange , df_cor_price_tilladelse$`Realiseret handelspris` )

### COVID 19 lows

In [ ]:
ggplot(df_udbud,aes(x=DATE,y=VALUE,color=ZIP)) +  geom_line(alpha=0.3) +  facet_wrap(~PROPERTY_TYPE)  +  scale_x_date(limits=c(ymd("2020-01-01"),NA),date_labels = "%b",date_breaks = "3 months")  +  geom_vline(data=df_udbud.covid19low,aes(xintercept=DATE,color=ZIP),linetype=2,alpha=0.7) +  theme_money_printer_go_brrr(base_size=12) 

### Bubble lows

In [ ]:
ggplot(df_udbud,aes(x=DATE,y=VALUE,color=ZIP)) +  geom_line(alpha=0.3) +  facet_wrap(~PROPERTY_TYPE)  +  scale_x_date(limits=c(ymd("2005-01-01"),ymd("2008-01-01")))  +  geom_vline(data=df_udbud.bubblelow,aes(xintercept=DATE,color=ZIP),linetype=2,alpha=0.7) +  theme_money_printer_go_brrr(base_size=12) 

## Construction Employment# Denmark House Prices

In [ ]:
df =  openxlsx::read.xlsx(xlsxFile = "Data/BM010.xlsx",startRow = 3) %>%   rename_with(~c("PROPERTY_TYPE","SALES_TYPE","ZIP"),1:3) %>%   tidyr::fill(PROPERTY_TYPE,.direction="down") %>%   tidyr::fill(SALES_TYPE,.direction="down") %>%   gather(DATE,VALUE,c(4:ncol(.))) %>%   filter(!VALUE=="..") %>%   dplyr::mutate(VALUE=as.integer(VALUE)) %>%   dplyr::mutate(DATE=lubridate::ceiling_date(as.Date(as.yearqtr(DATE, format = "%YK%q")),"quarters")-days(1))

In [ ]:
growth = function(data,indices) {  lm_ = lm(log(VALUE)~DATE,data=data[indices,])  return((((1+lm_$coefficients[2])^365)-1)*100)}coefs = function(data) {  ci_auc = tryCatch(expr = {    boot_ = boot(data=data,statistic=growth,R=1000)    ci_ = boot.ci(boot_, type="bca")    data.frame(t0=ci_$t0,                        lwr=ci_$bca[4],                        upr=ci_$bca[5])  }, error= function(e) {    data.frame()  })  return(ci_auc)}df_increase = df %>%   filter(PROPERTY_TYPE=="Realiseret handelspris") %>%   filter(VALUE>0) %>%   group_by(PROPERTY_TYPE,ZIP) %>%   do(coefs(data = .)) %>%   arrange(desc(t0))

## Increases Yearly Since 1992

In [ ]:
df_increase %>%   # filter(PROPERTY_TYPE=="Parcel-/rækkehus") %>%   arrange(t0) %>%   dplyr::mutate(ZIP=factor(ZIP,levels=.$ZIP)) %>%   ggplot(.,aes(x=ZIP,y=t0)) +  geom_point() +  geom_errorbar(aes(ymin=lwr,ymax=upr)) +  scale_y_continuous(limits=c(0,10)) +  geom_text(aes(label=paste0(scales::number(t0,accuracy = 0.1)," [",scales::number(lwr,accuracy = 0.1)," ; ",scales::number(upr,accuracy = 0.1),"]"),hjust=1.3),size=3) +  coord_flip() +  facet_wrap(~PROPERTY_TYPE)

## COVID-19 Increases### Houses

In [ ]:
days = as.integer( max(df$DATE) - ymd("2020-03-11"))growth = function(data,indices) {  lm_ = lm(log(VALUE)~DATE,data=data[indices,])  return((((1+lm_$coefficients[2])^days)-1)*100)}coefs_total = function(data) {  ci_auc = tryCatch(expr = {    boot_ = boot(data=data,statistic=growth,R=500)    ci_ = boot.ci(boot_, type="bca")    data.frame(t0=ci_$t0,                        lwr=ci_$bca[4],                        upr=ci_$bca[5])  }, error= function(e) {    data.frame()  })  return(ci_auc)}df_increase_dec_2019 = df %>%   filter(DATE>=ymd("2020-03-11")) %>%  # filter(PROPERTY_TYPE=="HOUSE") %>%   filter(PROPERTY_TYPE=="Realiseret handelspris") %>%   filter(VALUE>0) %>%   group_by(ZIP,SALES_TYPE) %>%   do(coefs_total(data = .)) %>%   arrange(desc(t0)) %>%    arrange(t0) %>%   dplyr::mutate(ZIP=factor(ZIP,levels=unique(.$ZIP)))df_increase_dec_2019 %>%   ggplot(.,aes(x=ZIP,y=t0)) +  geom_point() +  geom_errorbar(aes(ymin=lwr,ymax=upr)) +  scale_y_continuous(limits=c(0,100)) +  geom_text(aes(label=paste0(scales::number(t0,accuracy = 0.1)," [",scales::number(lwr,accuracy = 0.1)," ; ",scales::number(upr,accuracy = 0.1),"]"),hjust=1.3),size=3,vjust=-1,hjust=0) +  coord_flip() +  facet_wrap(~PROPERTY_TYPE)

## MARS Model

In [ ]:
hyper_grid <- expand.grid(  degree = seq(1,12,by=3),   nprune = seq(1, 101, length.out = 50) %>% floor()  ) %>% sample_n(30)df_mars = df %>%   filter(SALES_TYPE=="Realiseret handelspris") %>%   filter(PROPERTY_TYPE=="Parcel-/rækkehus") %>%   dplyr::filter(ZIP=="Lyngby-Taarbæk")tr_control = trainControl(method = "repeatedcv",number=5,repeats=5)fit_mars = caret::train(x = df_mars %>% dplyr::select(DATE),                        y=df_mars$VALUE,method="earth",                        tuneGrid=hyper_grid,                        metric="RMSE",                        trControl=tr_control)df_mars = df_mars %>% dplyr::mutate(pred=predict(fit_mars,.))break_dates = tail(unique(as.Date(summary(fit_mars$finalModel)$cuts[,1], origin = "1970-01-01")),-1)ggplot(df_mars,aes(x=DATE,y=pred)) +  # geom_line() +  geom_vline(xintercept = break_dates) +   annotate(geom = "label",x=break_dates,y = 10000,label=format(break_dates,"%Y %b"),size=2) +  geom_point(aes(y=VALUE)) +  scale_x_date(date_breaks = "1 year",date_labels = "%Y") 

## Financial Crisis

In [ ]:
df_finanskrise =   df %>%   filter(SALES_TYPE=="Realiseret handelspris") %>%   filter(PROPERTY_TYPE=="Parcel-/rækkehus") %>%   filter(VALUE>0) %>%   arrange(ZIP,DATE) %>%   filter(DATE<=ymd("2015-01-01")) %>%   group_by(ZIP) %>%   dplyr::mutate(Peak_Date=ifelse(VALUE==max(VALUE),                                DATE,                                NA)) %>%   tidyr::fill(Peak_Date,.direction="down") %>%   na.omit() %>%   dplyr::mutate(Max_M2_Price=max(VALUE)) %>%   dplyr::mutate(Trough_Date=ifelse(VALUE==min(VALUE),                                DATE,                                NA)) %>%   tidyr::fill(Peak_Date,.direction="up") %>%   na.omit() %>%   dplyr::mutate(Min_M2_Price=min(VALUE)) %>%   arrange(ZIP,DATE) %>%   dplyr::mutate(Days=as.integer(Trough_Date-Peak_Date)) %>%   dplyr::mutate(Pct_Chg=Min_M2_Price/Max_M2_Price) %>%   dplyr::mutate(Pct_Chg_Annualized=100*(Pct_Chg^(1/(Days/365.25))-1)) %>%   dplyr::mutate(Pct_Chg=100*(Pct_Chg-1))

### Losses by Zip Code

In [ ]:
df_finanskrise %>%   ggplot(.,aes(x=Max_M2_Price,y=Pct_Chg)) +  geom_point() +  geom_smooth() +  geom_label(aes(label=ZIP),size=4)

### Peak to Low

In [ ]:
df_finanskrise %>%   gather(date,val,Peak_Date,Trough_Date) %>%   dplyr::mutate(val=as.Date(as.numeric(val),origin="1970-01-01")) %>%   ggplot(.,aes(x=ZIP,y=val,group=ZIP)) +  geom_line() +  geom_point() +  coord_flip() +  scale_y_date(date_labels = "%Y",date_breaks = "1 year")

### Simulated if crash reoccurred

In [ ]:
current_max = max(df$DATE)df %>%   filter(SALES_TYPE=="Realiseret handelspris") %>%   filter(PROPERTY_TYPE=="Parcel-/rækkehus") %>%   filter(VALUE>0) %>%   group_by(ZIP) %>%   filter(DATE==max(DATE)) %>%   crossing(Length=seq(3)) %>%   dplyr::select(DATE,ZIP,PROPERTY_TYPE,VALUE,Length) %>%   inner_join(df_finanskrise %>% dplyr::select(ZIP,PROPERTY_TYPE,Pct_Chg_Annualized)) %>%   dplyr::mutate(VALUE=VALUE*(1+(Pct_Chg_Annualized)/100)^Length) %>%   dplyr::mutate(DATE=DATE+years(Length)) %>%   dplyr::select(DATE,ZIP,PROPERTY_TYPE,VALUE) %>%   bind_rows(df) %>%   dplyr::mutate(VALUE=VALUE*150) %>%   filter(DATE>=current_max) %>%   ggplot(.,aes(x=DATE,VALUE,color=ZIP)) +  geom_point(alpha=0.1) +   geom_line() +  geom_vline(aes(xintercept = DATE),linetype=3) +  geom_vline(xintercept = current_max,linetype=2) +  theme(legend.position="bottom") +  scale_y_continuous(labels = scales::number,breaks = seq(0,10000000,1000000))

### Long-term-trend

In [ ]:
df_non_bubble = df %>%   filter(SALES_TYPE=="Realiseret handelspris") %>%   filter(PROPERTY_TYPE=="Parcel-/rækkehus") %>%   filter(ZIP=="Lyngby-Taarbæk") %>%   filter((DATE >= ymd("1995-01-01") )) %>%   filter(!(DATE >= ymd("2005-01-01") & DATE<=ymd("2009-01-01"))) %>%   filter(!(DATE >= ymd("2020-09-01") ))lm_non_bubble = lm(VALUE~DATE,data=df_non_bubble)df_non_bubble = df %>%   filter(SALES_TYPE=="Realiseret handelspris") %>%   filter(PROPERTY_TYPE=="Parcel-/rækkehus") %>%   filter(ZIP=="Lyngby-Taarbæk") %>%   bind_rows(data.frame(DATE=seq.Date(from = max(df$DATE),to = max(df$DATE)+years(8),by="1 month"))) %>%   dplyr::mutate(pred=predict(lm_non_bubble,.)) last_bubble_length = ymd("2009-09-30")-ymd("2006-06-30")high_m2 = df_non_bubble %>% filter(DATE==max(df$DATE)) %>% na.omit(.) %>% pull(VALUE)normal_m2 = df_non_bubble %>% filter(DATE>=max(df$DATE)+days(last_bubble_length)) %>% head(1) %>%  pull(pred)normal_m2/high_m2df_non_bubble %>%   ggplot(.,aes(x=DATE,VALUE,color=ZIP)) +  geom_point(alpha=0.1) +   geom_line(aes(x=DATE,y=pred)) +  geom_line() +  scale_x_date(date_breaks = "1 year",date_labels = "%Y",limits = c(ymd("1992-01-01"),ymd("2028-01-01"))) +  scale_y_continuous(limits=c(0,50000)) +  geom_vline(xintercept = c(max(df$DATE))) +  geom_vline(xintercept = c(max(df$DATE)+days(last_bubble_length))) +  geom_vline(xintercept = ymd("2024-01-01"),linetype=2) +  geom_vline(xintercept = Sys.Date(),linetype=2) +  # geom_vline(aes(xintercept = DATE),linetype=3) +  # geom_vline(xintercept = current_max,linetype=2) +  theme(legend.position="bottom",        axis.text.x=element_text(angle=45,hjust=1)) #+  # scale_y_continuous(labels = scales::number,breaks = seq(0,10000000,1000000))df_non_bubble %>%   filter(DATE>=ymd("2019-01-01")) %>%   ggplot(.,aes(x=DATE,VALUE,color=ZIP)) +  geom_point(alpha=0.1) +   geom_line(aes(x=DATE,y=pred)) +  geom_line() +  scale_x_date(date_breaks = "1 year",date_labels = "%Y",limits = c(ymd("2019-01-01"),ymd("2026-01-01"))) +  scale_y_continuous(limits=c(33000,50000),breaks = seq(33000,50000,by=1000)) +  geom_vline(xintercept = c(max(df$DATE))) +  geom_vline(xintercept = c(max(df$DATE)+days(last_bubble_length))) +  geom_vline(xintercept = ymd("2024-01-01"),linetype=2) +  geom_vline(xintercept = Sys.Date(),linetype=2) +  geom_hline(yintercept = c(38700,39800)) +  # geom_vline(aes(xintercept = DATE),linetype=3) +  # geom_vline(xintercept = current_max,linetype=2) +  theme(legend.position="bottom",        axis.text.x=element_text(angle=45,hjust=1)) #+  # scale_y_continuous(labels = scales::number,breaks = seq(0,10000000,1000000))

## Adjustments

In [ ]:
df %>%   unite(SALES_TYPE,SALES_TYPE,PROPERTY_TYPE) %>%   spread(SALES_TYPE,VALUE) %>%   dplyr::mutate(ADJUST_HOUSE=(`Realiseret handelspris_Parcel-/rækkehus`/`Første udbudspris_Parcel-/rækkehus`)-1) %>%   # dplyr::mutate(ADJUST_FLAT=(`Realiseret handelspris_FLAT`/`Første udbudspris_FLAT`)-1) %>%   dplyr::select(ZIP,DATE,ADJUST_HOUSE) %>%   gather(ADJUST_TYPE,ADJUST_VAL,ADJUST_HOUSE) %>%   na.omit() %>%   separate(ADJUST_TYPE,into=c("SALES_TYPE","PROPERTY_TYPE")) %>%   ggplot(.,aes(x=DATE,y=ADJUST_VAL,color=ZIP,group=ZIP)) +  # geom_smooth() +  geom_line() +  theme(legend.position="bottom") +  scale_y_continuous(labels = scales::percent) +  scale_x_date(date_breaks = "2 year",date_labels = "%Y")

## Sales Times

In [ ]:
df_liggetider =  openxlsx::read.xlsx(xlsxFile = "Data/UDB030.xlsx",startRow = 3) %>%   dplyr::rename(TIME_TYPE=X2,                PROPERTY_TYPE=X1,                ZIP=X3) %>%   tidyr::fill(TIME_TYPE,.direction="down") %>%   tidyr::fill(PROPERTY_TYPE,.direction="down") %>%   filter(TIME_TYPE=="Liggetider (dage)") %>%   gather(DATE,VALUE,c(4:ncol(.))) %>%   filter(!VALUE=="..") %>%   dplyr::mutate(DATE=lubridate::ceiling_date(ymd(glue("{substr(DATE,1,4)}-{substr(DATE,6,7)}-01")),unit="month")-days(1))

In [ ]:
df_liggetider %>%   ggplot(.,aes(x=DATE,y=VALUE,color=ZIP,group=ZIP)) +  geom_line() +  scale_x_date(date_breaks = "2 year",date_labels = "%Y") +  scale_y_continuous(limits = c(0,NA))

## Supply

In [ ]:
df_udbud =  openxlsx::read.xlsx(xlsxFile = "Data/UDB010.xlsx",startRow = 3) %>%   dplyr::rename(SUPPLY_TYPE=X2,                PROPERTY_TYPE=X1,                ZIP=X3) %>%   tidyr::fill(SUPPLY_TYPE,.direction="down") %>%   tidyr::fill(PROPERTY_TYPE,.direction="down") %>%   tidyr::fill(ZIP,.direction="down") %>%   gather(DATE,VALUE,c(4:ncol(.))) %>%   filter(!VALUE=="..") %>%   dplyr::mutate(DATE=lubridate::ceiling_date(ymd(glue("{substr(DATE,1,4)}-{substr(DATE,6,7)}-01")),unit="month")-days(1))

In [ ]:
df_udbud %>%   ggplot(.,aes(x=DATE,y=VALUE,color=SUPPLY_TYPE,group=SUPPLY_TYPE)) +  geom_line() +  facet_wrap(~ZIP,scales="free")

In [ ]:
df_udbud %>%   ggplot(.,aes(x=DATE,y=VALUE,color=SUPPLY_TYPE,group=SUPPLY_TYPE)) +  geom_line() +  facet_wrap(~ZIP,scales="free")

Seasonality

In [ ]:
df_udbud %>%   # filter(ZIP=="Lyngby-Taarbæk") %>%  filter(ZIP=="Hele landet") %>%  # filter(DATE>=ymd("2019-01-01")) %>%   dplyr::mutate(floor_date=lubridate::floor_date(DATE,unit = "year")) %>%   group_by(PROPERTY_TYPE,SUPPLY_TYPE,ZIP,floor_date) %>%   arrange(PROPERTY_TYPE,SUPPLY_TYPE,ZIP,floor_date,DATE) %>%   dplyr::mutate(VALUE=VALUE-lag(VALUE)) %>%   filter(!is.na(VALUE)) %>%   dplyr::mutate(VALUE=cumsum(VALUE)) %>%   dplyr::mutate(floor_date=lubridate::floor_date(DATE,unit = "year")) %>%   dplyr::mutate(time=as.numeric(DATE-floor_date)) %>%   dplyr::mutate(floor_date=factor(year(floor_date))) %>%   ggplot(.,aes(x=time,y=VALUE,color=floor_date)) +  geom_line() +  facet_wrap(ZIP~SUPPLY_TYPE) +  theme(legend.position="bottom") +  scale_x_continuous(breaks = seq(30,12*30,by=30),labels = seq(12))

In [ ]:
df_udbud %>%   # filter(ZIP=="Lyngby-Taarbæk") %>%  filter(ZIP=="Hele landet") %>%  # filter(DATE>=ymd("2019-01-01")) %>%   dplyr::mutate(floor_date=lubridate::floor_date(DATE,unit = "year")) %>%   group_by(PROPERTY_TYPE,SUPPLY_TYPE,ZIP,floor_date) %>%   arrange(PROPERTY_TYPE,SUPPLY_TYPE,ZIP,floor_date,DATE) %>%   dplyr::mutate(floor_date=lubridate::floor_date(DATE,unit = "year"),                index_value=first(VALUE)) %>%   dplyr::mutate(index=100*VALUE/index_value,                time=as.numeric(DATE-floor_date)) %>%   dplyr::mutate(floor_date=factor(year(floor_date))) %>%   ggplot(.,aes(x=time,y=index,color=floor_date)) +  geom_line() +  facet_wrap(ZIP~SUPPLY_TYPE) +  theme(legend.position="bottom")

Months Supply

In [ ]:
df_sales = openxlsx::read.xlsx(xlsxFile = "Data/BM020.xlsx",startRow = 3) %>%   dplyr::rename(SALES_TYPE=X2,                PROPERTY_TYPE=X1,                ZIP=X3)  %>%   tidyr::fill(SALES_TYPE,.direction="down") %>%   tidyr::fill(PROPERTY_TYPE,.direction="down") %>%   gather(DATE,VALUE,c(4:ncol(.))) %>%   filter(!VALUE=="..") %>%  dplyr::mutate(DATE=lubridate::ceiling_date(as.Date(as.yearqtr(DATE, format = "%YK%q")),"quarters")-days(1)) %>%   filter(SALES_TYPE=="Solgte boliger")df_ms_udbud = df_udbud %>%   group_by(ZIP,PROPERTY_TYPE,SUPPLY_TYPE) %>%   arrange(ZIP,PROPERTY_TYPE,SUPPLY_TYPE,DATE) %>%   # dplyr::mutate(VALUE=VALUE-lag(VALUE)) %>%   rename(SALES_TYPE=SUPPLY_TYPE) %>%   spread(SALES_TYPE,VALUE) %>%   ungroup() %>%   dplyr::mutate(PROPERTY_TYPE=recode(PROPERTY_TYPE,"Parcel-/rækkehuse"="Parcel-/rækkehus"))df_ms_sales = df_sales %>%   group_by(ZIP,PROPERTY_TYPE,SALES_TYPE) %>%   arrange(ZIP,PROPERTY_TYPE,SALES_TYPE,DATE) %>%   spread(SALES_TYPE,VALUE) %>%   ungroup() df_ms_udbud %>%   inner_join( df_ms_sales ) %>%   dplyr::mutate(months_supply=`Udbudte boliger`/`Solgte boliger`) %>%   ggplot(.,aes(x=DATE,y=months_supply,color=ZIP)) +  geom_line()

## Insights into last peak

In [ ]:
source_url = "https://finansdanmark.dk/tal-og-data/boligstatistik/obligationsrenter/"current_url = xml2::read_html(source_url) %>%  html_node("body > main > div > div.page-header > div.page-header__content > div > div.row > div.col-12.col-md-8 > div > p:nth-child(11) > a") %>%   rvest::html_attr("href")xlsx_url = paste0("https://finansdanmark.dk/",current_url)

In [ ]:
df_interest = openxlsx::read.xlsx(xlsxFile = xlsx_url,startRow = 1)df_interest$År[1] = 1997df_interest = df_interest %>% fill(År,.direction = "down")df_interest$Date = as.Date(paste(df_interest$År, df_interest$Uge, 1, sep="-"), "%Y-%U-%u")df_interest = df_interest %>%   dplyr::select(Date,Kort.rente,Lang.rente) %>%   dplyr::mutate( CurveLongShort = Lang.rente - Kort.rente )

In [ ]:
BYGV80_meta <- dst_meta(table = "BYGV80", lang = "da")BYGV80 <- dst_get_data(table = "BYGV80",                        BYGFASE="*",                       ANVENDELSE="*",                       Tid="*",                       lang = "da")df_BYGV80 =   BYGV80 %>%   group_by(BYGFASE,ANVENDELSE) %>%   arrange(BYGFASE,ANVENDELSE,TID) %>%   filter(ANVENDELSE %in% c("Parcelhuse")) %>%   # filter(ANVENDELSE %in% c("Etageboliger","Parcelhuse")) %>%   # dplyr::mutate(ANVENDELSE=gsub("Etageboliger","FLAT",ANVENDELSE)) %>%   dplyr::mutate(ANVENDELSE=gsub("Parcelhuse","Parcel-/rækkehus",ANVENDELSE)) %>%   # dplyr::mutate(ANVENDELSE=gsub("Parcelhuse","Parcel-/rækkehus",ANVENDELSE)) %>%   dplyr::rename(PROPERTY_TYPE=ANVENDELSE) %>%   dplyr::rename(STAT=BYGFASE) %>%   dplyr::rename(DATE=TID) %>%   dplyr::rename(VALUE=value) %>%   na.omit()df_m2price =   df %>%   # filter(ZIP=="2800 Kgs.Lyngby") %>%   filter(SALES_TYPE=="Realiseret handelspris") %>%   dplyr::select(ZIP,DATE,SALES_TYPE,VALUE,PROPERTY_TYPE) %>%   dplyr::rename(STAT=SALES_TYPE)df_supply =   df_udbud %>%   # filter(ZIP=="Lyngby-Taarbæk") %>%   # dplyr::mutate(ZIP="2800 Kgs.Lyngby") %>%   dplyr::select(ZIP,DATE,SUPPLY_TYPE,VALUE,PROPERTY_TYPE) %>%   dplyr::rename(STAT=SUPPLY_TYPE)df_wait =   df_liggetider %>%   # filter(ZIP=="Lyngby-Taarbæk") %>%   # dplyr::mutate(ZIP="2800 Kgs.Lyngby") %>%   dplyr::select(ZIP,DATE,TIME_TYPE,VALUE,PROPERTY_TYPE) %>%   dplyr::rename(STAT=TIME_TYPE) 

In [ ]:
df_m2price %>%   bind_rows(df_supply) %>%  bind_rows(df_wait) %>%   bind_rows(df_BYGV80) %>%   bind_rows(df_interest) %>%   filter(ZIP=="Lyngby-Taarbæk") %>%   group_by(PROPERTY_TYPE,STAT) %>%   dplyr::mutate(scale_=scale(VALUE,center = TRUE,scale = TRUE)) %>%   # dplyr::mutate(scale_sma=TTR::SMA(scale_,n=10)) %>%   ggplot(.,aes(x=DATE,y=scale_,color=PROPERTY_TYPE)) +  geom_line() +  # geom_point(aes(x=DATE,y=scale_),size=0.3) +  facet_wrap(~STAT) +  theme(legend.position="bottom",        axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1)) +  scale_x_date(date_breaks = "3 year",date_labels = "%Y") +  geom_vline(xintercept = ymd("2022-03-31"),linetype=2) +  geom_vline(xintercept = ymd("2006-06-30"),linetype=2)

In [ ]:
df_m2price %>%   bind_rows(df_supply) %>%  bind_rows(df_wait) %>%   bind_rows(df_BYGV80) %>%   bind_rows(df_interest) %>%   filter(ZIP=="Lyngby-Taarbæk") %>%   crossing(data.frame(cycle=c("C1","C2"),dates=c(ymd("2006-06-30"),ymd("2022-03-31")))) %>%   filter(DATE>=dates) %>%   dplyr::mutate(DATE=as.numeric(DATE-dates)) %>%   group_by(cycle,PROPERTY_TYPE,STAT) %>%   arrange(cycle,PROPERTY_TYPE,STAT,DATE) %>%  dplyr::mutate(scale_=100*VALUE/first(VALUE)) %>%   # dplyr::mutate(scale_=scale(VALUE,center = TRUE,scale = TRUE)) %>%   # dplyr::mutate(scale_sma=TTR::SMA(scale_,n=10)) %>%   ggplot(.,aes(x=DATE,y=scale_,color=cycle)) +  geom_line() +  # geom_point(aes(x=DATE,y=scale_),size=0.3) +  facet_wrap(~STAT,scales="free") +  theme(legend.position="bottom",        axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1)) +  scale_x_continuous(breaks = seq(0,365*10,by=365),limits=c(0,365*10))

In [ ]:
df_m2price %>%   bind_rows(df_supply) %>%  bind_rows(df_wait) %>%   bind_rows(df_BYGV80) %>%   filter(ZIP=="Lyngby-Taarbæk") %>%   filter(STAT %in% c("Liggetider (dage)","Realiseret handelspris")) %>%   group_by(PROPERTY_TYPE,STAT) %>%   dplyr::mutate(scale_=scale(VALUE,center = TRUE,scale = TRUE)) %>%   dplyr::mutate(scale_sma=TTR::SMA(scale_,n=10)) %>%   ggplot(.,aes(x=DATE,y=scale_,color=STAT)) +  geom_line() +  # geom_point(aes(x=DATE,y=scale_),size=0.3) +  facet_wrap(~PROPERTY_TYPE) +  theme(legend.position="bottom",        axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1)) +  scale_x_date(date_breaks = "3 year",date_labels = "%Y") +  geom_vline(xintercept = c(ymd("2006-10-01"),ymd("2021-06-01")+days(457)),linetype=2) +  geom_vline(xintercept = c(ymd("2005-07-01"),ymd("2021-06-01")),linetype=2,color="red")

In [ ]:
df_m2price %>%   bind_rows(df_supply) %>%  bind_rows(df_wait) %>%   bind_rows(df_BYGV80) %>%   filter(ZIP=="Lyngby-Taarbæk") %>%   filter(STAT %in% c("Liggetider (dage)","Udbudte boliger")) %>%   group_by(PROPERTY_TYPE,STAT) %>%   dplyr::mutate(scale_=scale(VALUE,center = TRUE,scale = TRUE)) %>%   dplyr::mutate(scale_sma=TTR::SMA(scale_,n=10)) %>%   ggplot(.,aes(x=DATE,y=scale_,color=STAT)) +  geom_smooth(span=.7) +  # geom_point(aes(x=DATE,y=scale_),size=0.3) +  facet_wrap(~PROPERTY_TYPE) +  theme(legend.position="bottom",        axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1)) +  scale_x_date(date_breaks = "3 year",date_labels = "%Y") +  geom_vline(xintercept = c(ymd("2006-10-01"),ymd("2021-06-01")+days(457)),linetype=2) +  geom_vline(xintercept = c(ymd("2005-07-01"),ymd("2021-06-01")),linetype=2,color="red")

In [ ]:
df_m2price %>%   bind_rows(df_supply) %>%  bind_rows(df_wait) %>%   bind_rows(df_BYGV80) %>%   filter(ZIP=="Lyngby-Taarbæk") %>%   filter(STAT %in% c("Udbudte boliger","Realiseret handelspris")) %>%   group_by(PROPERTY_TYPE,STAT) %>%   dplyr::mutate(scale_=scale(VALUE,center = TRUE,scale = TRUE)) %>%   dplyr::mutate(scale_sma=TTR::SMA(scale_,n=10)) %>%   ggplot(.,aes(x=DATE,y=scale_,color=STAT)) +  geom_line() +  # geom_point(aes(x=DATE,y=scale_),size=0.3) +  facet_wrap(~PROPERTY_TYPE) +  theme(legend.position="bottom",        axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1)) +  scale_x_date(date_breaks = "3 year",date_labels = "%Y") +  geom_vline(xintercept = c(ymd("2006-10-01"),ymd("2021-06-01")+days(457)),linetype=2) +  geom_vline(xintercept = c(ymd("2005-07-01"),ymd("2021-06-01")),linetype=2,color="red")

In [ ]:
df_m2price %>%   bind_rows(df_supply) %>%  bind_rows(df_wait) %>%   filter(ZIP=="Hele landet") %>%   bind_rows(df_BYGV80) %>%   filter(STAT %in% c("Udbudte boliger","Byggeri under opførelse")) %>%   group_by(PROPERTY_TYPE,STAT) %>%   dplyr::mutate(scale_=scale(VALUE,center = TRUE,scale = TRUE)) %>%   dplyr::mutate(scale_sma=TTR::SMA(scale_,n=10)) %>%   ggplot(.,aes(x=DATE,y=scale_,color=STAT)) +  geom_line() +  # geom_point(aes(x=DATE,y=scale_),size=0.3) +  facet_wrap(~PROPERTY_TYPE) +  theme(legend.position="bottom",        axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1)) +  scale_x_date(date_breaks = "3 year",date_labels = "%Y") +  geom_vline(xintercept = c(ymd("2006-10-01"),ymd("2021-06-01")+days(457)),linetype=2) +  geom_vline(xintercept = c(ymd("2005-07-01"),ymd("2021-06-01")),linetype=2,color="red")

In [ ]:
df_m2price %>%   bind_rows(df_supply) %>%  bind_rows(df_wait) %>%   bind_rows(df_BYGV80) %>%   filter(ZIP=="Hele landet") %>%   bind_rows(df_interest %>% gather(STAT,VALUE,Kort.rente:CurveLongShort) %>% dplyr::rename(DATE=Date)) %>%   filter(STAT %in% c("Kort.rente","Realiseret handelspris")) %>%   group_by(PROPERTY_TYPE,STAT) %>%   dplyr::mutate(scale_=scale(VALUE,center = TRUE,scale = TRUE)) %>%   dplyr::mutate(scale_sma=TTR::SMA(scale_,n=10)) %>%   ggplot(.,aes(x=DATE,y=scale_,color=STAT)) +  geom_line() +  # geom_point(aes(x=DATE,y=scale_),size=0.3) +  facet_wrap(~PROPERTY_TYPE) +  theme(legend.position="bottom",        axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1)) +  scale_x_date(date_breaks = "3 year",date_labels = "%Y") +  geom_vline(xintercept = c(ymd("2006-10-01"),ymd("2021-06-01")+days(457)),linetype=2) +  geom_vline(xintercept = c(ymd("2005-07-01"),ymd("2021-06-01")),linetype=2,color="red")

In [ ]:
df_m2price %>%   bind_rows(df_supply) %>%  bind_rows(df_wait) %>%   bind_rows(df_BYGV80) %>%   bind_rows(df_interest) %>%   filter(STAT %in% c("Kort.rente","Lang.rente")) %>%   group_by(PROPERTY_TYPE,STAT) %>%   dplyr::mutate(scale_=scale(VALUE,center = TRUE,scale = TRUE)) %>%   dplyr::mutate(scale_sma=TTR::SMA(scale_,n=10)) %>%   ggplot(.,aes(x=DATE,y=scale_,color=STAT)) +  geom_line() +  # geom_point(aes(x=DATE,y=scale_),size=0.3) +  facet_wrap(~PROPERTY_TYPE) +  theme(legend.position="bottom",        axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1)) +  scale_x_date(date_breaks = "3 year",date_labels = "%Y") +  geom_vline(xintercept = c(ymd("2006-10-01"),ymd("2021-06-01")+days(457)),linetype=2) +  geom_vline(xintercept = c(ymd("2005-07-01"),ymd("2021-06-01")),linetype=2,color="red")

# Denmark: Building Occupation

In [ ]:
BOL102_meta <- dst_meta(table = "BOL102", lang = "da")BOL102 <- dst_get_data(table = "BOL102",                        AMT="*",                       BEBO="*",                       ANVENDELSE="*",                       OPFØRELSESÅR="*",                       OPVARMNING="*",                       TOILET="*",                       BAD="*",                       KOEKKEN="*",                       Tid="*",                       lang = "da")

In [ ]:
BOL101 %>%   filter(OMRÅDE=="Hele landet") %>%   filter(BEBO=="Boliger uden CPR tilmeldte personer (ubeboede boliger)") %>%   filter(ANVENDELSE=="Parcel/Stuehuse") %>%   group_by( OMRÅDE , BEBO , ANVENDELSE , TID) %>%   dplyr::summarize(count=sum(value)) %>%   ggplot(.,aes(x=TID,y=count)) +  geom_line()

In [ ]:
df_BOL101 =   BOL101 %>%   group_by(OMRÅDE,TID,ANVENDELSE,BEBO) %>%   dplyr::summarise(value=sum(value)) %>%   # dplyr::mutate(value=scale(value)) %>%   ungroup()df_BOL101 %>%   filter(OMRÅDE=="Hele landet") %>%   ggplot(.,aes(x=TID,y=value,color=BEBO)) +  geom_line() +  facet_wrap(~ANVENDELSE+BEBO,scales="free") +  theme(legend.position="bottom") +  theme_money_printer_go_brrr(base_size=12) 

## Ratio

In [ ]:
df_BOL101 =   BOL101 %>%   group_by(OMRÅDE,TID,ANVENDELSE,BEBO) %>%   dplyr::summarise(value=sum(value)) %>%   # dplyr::mutate(value=scale(value)) %>%   ungroup()df_BOL101 %>%   spread(BEBO,value) %>%   dplyr::mutate(ratio=`Boliger med CPR tilmeldte personer (beboede boliger)`/`Boliger uden CPR tilmeldte personer (ubeboede boliger)`) %>%   ggplot(.,aes(x=TID,y=ratio,color=ANVENDELSE)) +  geom_line() +  facet_wrap(~OMRÅDE,scales="free") +  theme(legend.position="bottom") +  theme_money_printer_go_brrr(base_size=12) 

In [ ]:
df =  openxlsx::read.xlsx(xlsxFile = "Data/BM010.xlsx",startRow = 3) %>%   rename_with(~c("PROPERTY_TYPE","SALES_TYPE","ZIP"),1:3) %>%   tidyr::fill(PROPERTY_TYPE,.direction="down") %>%   tidyr::fill(SALES_TYPE,.direction="down") %>%   tidyr::fill(ZIP,.direction="down") %>%   gather(DATE,VALUE,c(4:ncol(.))) %>%   filter(!VALUE=="..") %>%   dplyr::mutate(VALUE=as.integer(VALUE)) %>%   dplyr::mutate(DATE=lubridate::ceiling_date(as.Date(as.yearqtr(DATE, format = "%YK%q")),"quarters")-days(1))df_m2price =   df %>%   # filter(ZIP=="2800 Kgs.Lyngby") %>%   filter(SALES_TYPE=="Realiseret handelspris") %>%   dplyr::select(ZIP,DATE,SALES_TYPE,VALUE,PROPERTY_TYPE) %>%   dplyr::rename(STAT=SALES_TYPE)

In [ ]:
source_url = "https://finansdanmark.dk/tal-og-data/boligstatistik/obligationsrenter/"current_url = xml2::read_html(source_url) %>%  html_node("body > main > div > div.page-header > div.page-header__content > div > div.row > div.col-12.col-md-8 > div > p:nth-child(11) > a") %>%   rvest::html_attr("href")xlsx_url = paste0("https://finansdanmark.dk/",current_url)df_interest = openxlsx::read.xlsx(xlsxFile = xlsx_url,startRow = 1)df_interest$År[1] = 1997df_interest = df_interest %>% fill(År,.direction = "down")df_interest$Date = as.Date(paste(df_interest$År, df_interest$Uge, 1, sep="-"), "%Y-%U-%u")df_interest =   df_interest %>%   dplyr::select(Date,Kort.rente,Lang.rente) %>%   dplyr::mutate( CurveLongShort = Lang.rente - Kort.rente ) %>%   gather(RATE_TYPE,RATE_VALUE,Kort.rente:CurveLongShort) %>%   dplyr::rename(DATE=Date) %>%   group_by(RATE_TYPE) %>%   tidyr::complete(DATE = seq.Date(from = min(DATE,na.rm=T), to = max(DATE,na.rm=T), by="day")) %>%   tidyr::fill(RATE_VALUE,.direction="downup") 

In [ ]:
df_m2price %>%   filter(ZIP %in% c("Lyngby-Taarbæk")) %>%   inner_join( df_interest ) %>%   filter(RATE_TYPE %in% c("Kort.rente","Lang.rente")) %>%   dplyr::mutate(RATE_BURDEN=VALUE*(RATE_VALUE/100)) %>%   ggplot(.,aes(x=DATE,y=RATE_BURDEN,color=PROPERTY_TYPE)) +  geom_line() +  facet_wrap(~RATE_TYPE,ncol=1) +  scale_x_date(date_breaks = "1 year",date_labels = "%Y") +  geom_hline(yintercept = 0,linetype=2) +  theme_money_printer_go_brrr(base_size=12)  +  labs(title="Boligbyrde: DK Interest Payment per SQMT",       x="Date",       y="DKK",       caption = timestamp_caption()       )

## Household income 30%

In [ ]:
#```{r }df_m2price %>%   filter(ZIP %in% c("Lyngby-Taarbæk")) %>%   inner_join( df_interest ) %>%   filter(RATE_TYPE %in% c("Kort.rente","Lang.rente")) %>%   dplyr::mutate(RATE_BURDEN=180*VALUE*(RATE_VALUE/100)) %>%   dplyr::mutate(RATE_BURDEN=RATE_BURDEN/(1/3)) %>%   ggplot(.,aes(x=DATE,y=RATE_BURDEN,color=PROPERTY_TYPE)) +  geom_line() +  facet_wrap(~RATE_TYPE,ncol=1) +  scale_x_date(date_breaks = "1 year",date_labels = "%Y") +  geom_hline(yintercept = 0,linetype=2) +  theme_money_printer_go_brrr(base_size=12)  +  labs(title="Boligbyrde: DK Interest Payment per SQMT",       x="Date",       y="DKK",       caption = timestamp_caption()       )

# Twin Peaks

In [ ]:
# EJ5EJ5_meta <- dst_meta(table = "EJ5", lang = "da")EJ5 <- dst_get_data(table = "EJ5",                        EJENDOMSKATE="Enfamiliehuse",                       TAL="Indeks",                       Tid="*",                       lang = "da") %>%   dplyr::mutate(series_id=EJ5_meta$basics$id) %>%   tidyr::complete(TID = seq.Date(from = min(TID,na.rm=T), to = max(TID,na.rm=T), by="day")) %>%   tidyr::fill(value,.direction="downup")  %>%   tidyr::fill(TAL,.direction="downup")   %>%   tidyr::fill(EJENDOMSKATE,.direction="downup")    %>%   tidyr::fill(series_id,.direction="downup") %>%   dplyr::rename(date=TID,                value=value)# BYGV80BYGV80_meta <- dst_meta(table = "BYGV80", lang = "da")BYGV80 <- dst_get_data(table = "BYGV80",                        ANVENDELSE="*",                       BYGFASE="*",                       Tid="*",                       lang = "da") %>%   filter(ANVENDELSE %in% c("Parcelhuse","Række-, kæde- og dobbelthuse")) %>%   group_by(BYGFASE,TID) %>%   dplyr::summarize(value=sum(value,na.rm=T)) %>%   ungroup() %>%   dplyr::mutate(ANVENDELSE="Enfamiliehus") %>%   dplyr::mutate(series_id=BYGV80$basics$id) %>%   tidyr::complete(TID = seq.Date(from = min(TID,na.rm=T), to = max(TID,na.rm=T), by="day")) %>%   tidyr::fill(value,.direction="downup")  %>%   tidyr::fill(ANVENDELSE,.direction="downup")    %>%   tidyr::fill(BYGFASE,.direction="downup")    %>%   tidyr::fill(series_id,.direction="downup") %>%   dplyr::rename(date=TID,                value=value)# %>%   dplyr::mutate(series_id=EJ5_meta$basics$id) %>%   tidyr::complete(TID = seq.Date(from = min(TID,na.rm=T), to = max(TID,na.rm=T), by="day")) %>%   tidyr::fill(value,.direction="downup")  %>%   tidyr::fill(TAL,.direction="downup")   %>%   tidyr::fill(EJENDOMSKATE,.direction="downup")    %>%   tidyr::fill(series_id,.direction="downup") %>%   dplyr::rename(date=TID,                value=value)

In [ ]:
# Loopdf_CSUSHPINSA = do.call("rbind",list(EJ5)) %>%   crossing(data.frame(peak=c("2007","2022"),           peak_date=c(ymd("2007-07-01",ymd("2022-04-01"))))) %>%   filter(date>=peak_date-years(3)) %>%   filter(date<=peak_date+years(7)) %>%   arrange(peak,peak_date,date) %>%   group_by(peak,series_id) %>%   arrange(peak,series_id,date) %>%   dplyr::mutate(index=ifelse(date==peak_date,value,NA),                index_diffdate=date-peak_date) %>%   dplyr::mutate(label=ifelse(index_diffdate==max(index_diffdate),series_id,NA)) %>%   tidyr::fill(index,.direction = "updown") %>%   dplyr::mutate(index=100*value/index)

In [ ]:
df_CSUSHPINSA %>%   ggplot(.,aes(x=index_diffdate,y=index,color=series_id)) +  geom_line() +  # facet_wrap(~peak,ncol=1,scales="free_x") +  facet_wrap(~peak,ncol=1,scales="free_y") +  # geom_text_repel(aes(label=label),nudge_x = 0, direction = "y", hjust = "left") +  geom_text(aes(label=label),nudge_x = 0, direction = "y", hjust = "left") +  geom_hline(yintercept = 100,linetype=2) +  geom_vline(xintercept = 0,linetype=2) +  scale_x_continuous(expand=expansion(add=c(0,1000))) +  scale_x_continuous(breaks = seq(-3,7)*365,labels=seq(-3,7))  # geom_vline(aes(xintercept = peak_date))

In [ ]:
df_CSUSHPINSA %>%   ggplot(.,aes(x=index_diffdate,y=index,color=peak)) +  geom_line() +  # facet_wrap(~peak,ncol=1,scales="free_x") +  facet_wrap(~series_id,scales="free_y") +  # geom_text_repel(aes(label=label),nudge_x = 0, direction = "y", hjust = "left") +  geom_text(aes(label=label),nudge_x = 0, direction = "y", hjust = "left") +  scale_x_continuous(expand=expansion(add=c(0,1000))) +  geom_hline(yintercept = 100,linetype=2) +  geom_vline(xintercept = 0,linetype=2) +  scale_x_continuous(breaks = seq(-3,7)*365,labels=seq(-3,7))  # geom_vline(aes(xintercept = peak_date))